#analise do comportamento de clientes#

In [0]:
## CRIANDO TABELA UNIFICADA: SCORE GLOBAL + ESTADO + CIDADE

from pyspark.sql.functions import col, count, sum as spark_sum, lit, log10, percent_rank, when
from pyspark.sql.window import Window

# Configuração
CATALOG = "workspace"
SCHEMA_SILVER = "yelp_silver"
SCHEMA_GOLD = "yelp_gold"

print("Carregando tabelas...")
print("="*60)

# Carrega tabelas necessárias
df_user = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.user")
df_review = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.review")
df_business = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.business")

from pyspark.sql.functions import date_format, to_date

df_user = df_user.withColumn(
    "yelping_since",
    date_format(to_date(col("yelping_since")), "MM/yyyy")
)

# ============================================
# 1. SCORE GLOBAL (uma linha por usuário)
# ============================================

print("Calculando score GLOBAL...")
print("="*60)

# Agregação global por user_id
df_global = (
    df_review
    .join(df_user.select("user_id", "name", "fans", "average_stars", "yelping_since", "elite"), "user_id", "inner")
    .groupBy("user_id", "name", "fans", "average_stars", "yelping_since", "elite")
    .agg(
        count("review_id").alias("review_food_count"),
        spark_sum("cool").alias("review_cool"),
        spark_sum("funny").alias("review_funny"),
        spark_sum("useful").alias("review_useful"),
        # Contagem por categoria de comida
        spark_sum(when(col("food_category") == "RESTAURANTE", 1).otherwise(0)).alias("reviews_restaurante"),
        spark_sum(when(col("food_category") == "BAR E BEBIDA", 1).otherwise(0)).alias("reviews_bar_bebida"),
        spark_sum(when(col("food_category") == "CAFE", 1).otherwise(0)).alias("reviews_cafe"),
        spark_sum(when(col("food_category") == "PADARIA", 1).otherwise(0)).alias("reviews_padaria")
    )
)

# Adiciona coluna booleana indicando se elite está preenchida
df_global = df_global.withColumn("has_elite", col("elite").isNotNull())

# Calcula score global com escala logarítmica
df_global = df_global.withColumn(
    "score",
    (log10(col("review_food_count") + 1) * 70) + 
    (log10(col("fans") + 1) * 20) +
    (log10((col("review_cool") + col("review_funny") + col("review_useful")) + 1) * 10)
)

# Adiciona colunas de localidade como NULL (marca como global)
df_global = df_global.withColumn("state", lit(None).cast("string")) \
                     .withColumn("city", lit(None).cast("string")) \
                     .withColumn("scope", lit("global"))


print(f"Total de usuários (global): {df_global.count()}")

# ============================================
# 2. SCORE POR ESTADO (múltiplas linhas por usuário)
# ============================================

print("Calculando score por ESTADO...")
print("="*60)

# JOIN: user -> review -> business (para pegar state)
df_state = (
    df_review
    .join(df_business.select("business_id", "state"), "business_id", "inner")
    .join(df_user.select("user_id", "name", "fans", "average_stars", "yelping_since", "elite"), "user_id", "inner")
    .filter(col("state").isNotNull())
)

# Agregação por user_id + state (SEM city)
df_state = (
    df_state
    .groupBy("user_id", "name", "state", "fans", "average_stars", "yelping_since", "elite")
    .agg(
        count("review_id").alias("review_food_count"),
        spark_sum("cool").alias("review_cool"),
        spark_sum("funny").alias("review_funny"),
        spark_sum("useful").alias("review_useful"),
        # Contagem por categoria de comida
        spark_sum(when(col("food_category") == "RESTAURANTE", 1).otherwise(0)).alias("reviews_restaurante"),
        spark_sum(when(col("food_category") == "BAR E BEBIDA", 1).otherwise(0)).alias("reviews_bar_bebida"),
        spark_sum(when(col("food_category") == "CAFE", 1).otherwise(0)).alias("reviews_cafe"),
        spark_sum(when(col("food_category") == "PADARIA", 1).otherwise(0)).alias("reviews_padaria")
    )
)

# Calcula score por estado
df_state = df_state.withColumn(
    "score",
    (log10(col("review_food_count") + 1) * 70) + 
    (log10(col("fans") + 1) * 20) +
    (log10((col("review_cool") + col("review_funny") + col("review_useful")) + 1) * 10)
)

df_state = df_state.withColumn("city", lit(None).cast("string")) \
                   .withColumn("scope", lit("state"))

print(f"Total de registros (por estado): {df_state.count()}")

# ============================================
# 3. SCORE POR CIDADE (múltiplas linhas por usuário)
# ============================================

print("Calculando score por CIDADE...")
print("="*60)

# JOIN: user -> review -> business (para pegar city/state)
df_city = (
    df_review
    .join(df_business.select("business_id", "city", "state"), "business_id", "inner")
    .join(df_user.select("user_id", "name", "fans", "average_stars", "yelping_since", "elite"), "user_id", "inner")
    .filter((col("city").isNotNull()) & (col("state").isNotNull()))
)

# Agregação por user_id + state + city
df_city = (
    df_city
    .groupBy("user_id", "name", "state", "city", "fans", "average_stars", "yelping_since", "elite")
    .agg(
        count("review_id").alias("review_food_count"),
        spark_sum("cool").alias("review_cool"),
        spark_sum("funny").alias("review_funny"),
        spark_sum("useful").alias("review_useful"),
        # Contagem por categoria de comida
        spark_sum(when(col("food_category") == "RESTAURANTE", 1).otherwise(0)).alias("reviews_restaurante"),
        spark_sum(when(col("food_category") == "BAR E BEBIDA", 1).otherwise(0)).alias("reviews_bar_bebida"),
        spark_sum(when(col("food_category") == "CAFE", 1).otherwise(0)).alias("reviews_cafe"),
        spark_sum(when(col("food_category") == "PADARIA", 1).otherwise(0)).alias("reviews_padaria")
    )
)

# Calcula score por cidade
df_city = df_city.withColumn(
    "score",
    (log10(col("review_food_count") + 1) * 70) + 
    (log10(col("fans") + 1) * 20) +
    (log10((col("review_cool") + col("review_funny") + col("review_useful")) + 1) * 10)
)

df_city = df_city.withColumn("scope", lit("city"))

print(f"Total de registros (por cidade): {df_city.count()}")

# ============================================
# 4. UNIÃO: GLOBAL + ESTADO + CIDADE
# ============================================

print("Unindo dados globais, por estado e por cidade...")
print("="*60)

# Seleciona mesmas colunas em todos
cols_final = [
    "user_id", "name", "state", "city", "scope",
    "score", "review_food_count", "review_cool", "review_funny", "review_useful",
    "reviews_restaurante", "reviews_bar_bebida", "reviews_cafe", "reviews_padaria",
    "fans", "average_stars", "yelping_since", "elite"
]

df_final = df_global.select(cols_final) \
                    .union(df_state.select(cols_final)) \
                    .union(df_city.select(cols_final))

# ============================================
# 5. MARCANDO TOP 20% EM CADA ÂMBITO
# ============================================

print("Marcando top 20% em cada âmbito...")
print("="*60)

# Define window specs para cada âmbito
window_global = Window.partitionBy("scope").orderBy(col("score").desc())
window_state = Window.partitionBy("scope", "state").orderBy(col("score").desc())
window_city = Window.partitionBy("scope", "state", "city").orderBy(col("score").desc())

# Calcula percent_rank para cada registro no contexto apropriado
df_final = df_final.withColumn(
    "percent_rank_value",
    when(col("scope") == "global", percent_rank().over(window_global))
    .when(col("scope") == "state", percent_rank().over(window_state))
    .when(col("scope") == "city", percent_rank().over(window_city))
)

# Marca como top 20% se percent_rank <= 0.20 (top 20%)
df_final = df_final.withColumn(
    "is_top_20_percent",
    (col("percent_rank_value") <= 0.20).cast("boolean")
)

# Remove coluna auxiliar
df_final = df_final.drop("percent_rank_value")



print("Salvando tabela user_behavior_complete...")
print("="*60)

df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.user_behavior_complete"
)

print("✅ Tabela criada com sucesso!")
print(f"Total de registros: {df_final.count()}")
print(f"Total de registros top 20%: {df_final.filter(col('is_top_20_percent')).count()}")
print("\n🌍 Top 10 GLOBAL:")
display(df_final.filter(col("scope") == "global").orderBy(col("score").desc()).limit(10))
print("\n🗺️ Top 10 em PA (estado):")
display(df_final.filter((col("scope") == "state") & (col("state") == "PA")).orderBy(col("score").desc()).limit(10))
print("\n🏙️ Top 10 em Philadelphia (cidade):")
display(df_final.filter((col("scope") == "city") & (col("city") == "PHILADELPHIA")).orderBy(col("score").desc()).limit(10))

## 📊 Como Usar a Tabela no Dashboard

A tabela `user_behavior_complete` foi estruturada para facilitar o uso no dashboard com um único widget e **3 níveis de granularidade**.

### Estrutura da Tabela

| scope | state | city | is_top_20_percent | Descrição |
|-------|-------|------|-------------------|------------|
| `'global'` | NULL | NULL | TRUE/FALSE | Score global + marca se está nos top 20% globais |
| `'state'` | 'PA' | NULL | TRUE/FALSE | Score estadual + marca se está nos top 20% daquele estado |
| `'city'` | 'PA' | 'Philadelphia' | TRUE/FALSE | Score cidade + marca se está nos top 20% daquela cidade |

**⭐ Nova coluna:** `is_top_20_percent` marca automaticamente os top 20% usuários em cada âmbito usando percentil ranking.

### Queries para o Dashboard

#### 1️⃣ **Top 10 Usuários GLOBAIS**
```sql
SELECT name, score, review_food_count, fans, is_top_20_percent
FROM workspace.yelp_gold.user_behavior_complete
WHERE scope = 'global'
ORDER BY score DESC
LIMIT 10
```

#### 2️⃣ **Top 20% Usuários por ESTADO**
```sql
SELECT name, state, score, review_food_count, fans
FROM workspace.yelp_gold.user_behavior_complete
WHERE scope = 'state' 
  AND state = :state_param
  AND is_top_20_percent = TRUE
ORDER BY score DESC
```

#### 3️⃣ **Top 20% Usuários por CIDADE**
```sql
SELECT name, state, city, score, review_food_count, fans
FROM workspace.yelp_gold.user_behavior_complete
WHERE scope = 'city' 
  AND state = :state_param 
  AND city = :city_param
  AND is_top_20_percent = TRUE
ORDER BY score DESC
```

#### 4️⃣ **Contagem de Top 20% por Estado**
```sql
SELECT 
  state,
  COUNT(*) as total_users,
  SUM(CASE WHEN is_top_20_percent THEN 1 ELSE 0 END) as top_20_count
FROM workspace.yelp_gold.user_behavior_complete
WHERE scope = 'state'
GROUP BY state
ORDER BY top_20_count DESC
```

#### 5️⃣ **Widget Único com Filtro Dinâmico de 3 Níveis**
```sql
SELECT 
  name AS "Nome",
  COALESCE(state, 'Global') AS "Estado",
  COALESCE(city, '-') AS "Cidade",
  CAST(score AS INT) AS "Score",
  review_food_count AS "Reviews",
  fans AS "Fãs",
  CASE WHEN is_top_20_percent THEN '⭐ Top 20%' ELSE '' END AS "Ranking"
FROM workspace.yelp_gold.user_behavior_complete
WHERE 
  CASE 
    WHEN :granularity = 'global' THEN scope = 'global'
    WHEN :granularity = 'state' THEN scope = 'state' AND state = :location
    WHEN :granularity = 'city' THEN scope = 'city' AND city = :location
  END
ORDER BY score DESC
LIMIT 20
```

### Vantagens dessa Estrutura

✅ **Três níveis de análise** → Global, Estado, Cidade  
✅ **Marcação automática de top 20%** → Calculado por percentil em cada âmbito  
✅ **Um único dataset** → Mais fácil de manter  
✅ **Mesmo widget** → Usa parâmetros para alternar entre visões  
✅ **Comparações fáceis** → Pode mostrar score global vs estadual vs cidade lado a lado  
✅ **Flexível** → Filtros dinâmicos sem mudar estrutura  

### Como funciona o Top 20%

* **Global**: Top 20% de TODOS os usuários da plataforma
* **Estado**: Top 20% dos usuários DAQUELE estado específico  
* **Cidade**: Top 20% dos usuários DAQUELA cidade específica

Um usuário pode estar no top 20% em um âmbito mas não em outro!

### Exemplo de Uso no Dashboard

**Parâmetros sugeridos:**
* `granularity`: dropdown ['global', 'state', 'city']
* `location`: text input ou dropdown dinâmico
* `show_only_top_20`: checkbox [TRUE/FALSE] para filtrar apenas top 20%

In [0]:
%sql
-- Query de exemplo para usar no dashboard
-- Demonstra os 3 níveis: GLOBAL, ESTADO, CIDADE + marcação de TOP 20%

-- 🌍 1. TOP 10 GLOBAL (com marca de top 20%)
SELECT 
  name AS "Nome do Usuário",
  'Global' AS "Granularidade",
  CAST(score AS INT) AS "Score",
  review_food_count AS "Reviews",
  fans AS "Fãs",
  CASE WHEN is_top_20_percent THEN '⭐ Top 20%' ELSE '' END AS "Ranking"
FROM workspace.yelp_gold.user_behavior_complete
WHERE scope = 'global'
ORDER BY score DESC
LIMIT 10;

-- 🗺️ 2. SOMENTE TOP 20% EM PENNSYLVANIA (estado)
SELECT 
  name AS "Nome do Usuário",
  state AS "Estado",
  CAST(score AS INT) AS "Score",
  review_food_count AS "Reviews",
  fans AS "Fãs"
FROM workspace.yelp_gold.user_behavior_complete
WHERE scope = 'state' 
  AND state = 'PA'
  AND is_top_20_percent = TRUE
ORDER BY score DESC;

-- 🏙️ 3. SOMENTE TOP 20% EM PHILADELPHIA (cidade)
SELECT 
  name AS "Nome do Usuário",
  state AS "Estado",
  city AS "Cidade",
  CAST(score AS INT) AS "Score",
  review_food_count AS "Reviews"
FROM workspace.yelp_gold.user_behavior_complete
WHERE scope = 'city' 
  AND state = 'PA' 
  AND city = 'Philadelphia'
  AND is_top_20_percent = TRUE
ORDER BY score DESC;

-- 📊 4. ESTATÍSTICAS: Quantidade de Top 20% por Estado
SELECT 
  state AS "Estado",
  COUNT(*) AS "Total Usuários",
  SUM(CASE WHEN is_top_20_percent THEN 1 ELSE 0 END) AS "Top 20% Count",
  ROUND(SUM(CASE WHEN is_top_20_percent THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS "% Top 20%"
FROM workspace.yelp_gold.user_behavior_complete
WHERE scope = 'state'
GROUP BY state
ORDER BY "Top 20% Count" DESC;